# Dự đoán giá bất động sản TP.HCM (cũ) — từ dữ liệu thô đến mô hình

**Mục tiêu.** Dự đoán **giá/m²** (triệu đồng) của tin rao bán nhà đất tại TP.HCM cũ từ các thuộc tính
của tin: loại nhà đất, quận, diện tích, mặt tiền, phòng ngủ, số tầng, tọa độ và khoảng cách tới trung tâm.

**Quy trình.**

| # | Bước | Nội dung |
|---|---|---|
| 1 | Cấu hình | Thư viện, hằng số, tên cột |
| 2 | Nạp dữ liệu | Đọc CSV huấn luyện, kiểm tra schema |
| 3 | Kiểm tra chất lượng | Thiếu giá trị, trùng lặp, tọa độ |
| 4 | Tiền xử lý | Chuẩn hóa tọa độ, giải mã nhãn, loại tin không có tọa độ |
| 5 | EDA | Phân phối, quan hệ với giá, bản đồ, tương quan |
| 6 | Feature engineering | Khoảng cách tới trung tâm (Haversine), cờ thiếu dữ liệu, pháp lý |
| 7 | Chia train/test | Phân tầng theo khoảng cách trung tâm |
| 8 | Xử lý ngoại lai | So sánh IQR với luật nghiệp vụ, chỉ áp cho train |
| 9 | Huấn luyện & so sánh | 7 mô hình, cross-validation, chọn mô hình và bộ đặc trưng |
| 10 | Tinh chỉnh | RandomizedSearchCV cho mô hình tốt nhất |
| 11 | Đánh giá cuối | Chỉ số trên test, phân tích sai số, độ quan trọng đặc trưng |
| 12 | Lưu mô hình | joblib + metadata |

**Nguyên tắc chống rò rỉ dữ liệu (data leakage).**
- Không dùng cột `giá` làm đặc trưng vì `giá = giá/m² × diện tích`, tức là đã chứa đáp án.
- Chia train/test **trước** khi điền thiếu và loại ngoại lai; mọi thống kê (median, ngưỡng) chỉ tính trên train.
- Tập test giữ nguyên phân phối thật (không loại ngoại lai) để số đo phản ánh dữ liệu thực tế.

## 1. Cấu hình

In [ ]:
import warnings
from datetime import datetime
from pathlib import Path

import hashlib
import importlib.metadata

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (
    KFold, RandomizedSearchCV, StratifiedShuffleSplit, cross_val_predict, cross_validate, learning_curve,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12, "axes.labelsize": 10})
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
pd.set_option("display.max_columns", 30)

SEED = 42
DATA_PATH = Path("data/price_training_db.csv")
MODEL_DIR = Path("models_exp")
SAVE_MODEL = True            # đặt False để chạy thử mà không ghi file

TARGET = "giá/m2"
CENTER_LAT, CENTER_LON = 10.7769, 106.7009    # Chợ Bến Thành / trung tâm TP.HCM
HCM_LAT, HCM_LON = (10.3, 11.2), (106.3, 107.1)  # khung tọa độ TP.HCM cũ (cùng khung với pipeline làm sạch)
COORD_SCALE = 1_000_000_000                   # CSV lưu tọa độ nhân 10^9

PROPERTY_TYPES = {
    0: "căn hộ chung cư", 1: "chung cư mini, căn hộ dịch vụ", 2: "nhà riêng",
    3: "nhà biệt thự, liền kề", 4: "nhà mặt phố", 5: "shophouse, nhà phố thương mại",
    6: "đất nền dự án", 7: "bán đất", 8: "condotel", 9: "kho nhà xưởng",
}
DISTRICTS = dict(enumerate([
    "bình chánh", "bình tân", "bình thạnh", "cần giờ", "củ chi", "gò vấp", "hóc môn", "nhà bè",
    "phú nhuận", "quận 1", "quận 10", "quận 11", "quận 12", "quận 2", "quận 3", "quận 4",
    "quận 5", "quận 6", "quận 7", "quận 8", "quận 9", "thủ đức", "tân bình", "tân phú",
]))

# Đặc trưng cơ bản (giống mô hình backend). Không có "giá" (rò rỉ đáp án).
BASE_FEATURES = ["loại nhà đất", "địa chỉ", "diện tích", "mặt tiền", "phòng ngủ",
                 "tọa độ x", "tọa độ y", "số tầng", "cách trung tâm"]
FILL_COLUMNS = ["số tầng", "phòng ngủ", "mặt tiền"]
# Đặc trưng bổ sung (bước 6.2): cờ "ô bị thiếu" + pháp lý (0 = không rõ)
MISSING_FLAGS = [f"thiếu {c}" for c in FILL_COLUMNS]
EXTRA_FEATURES = MISSING_FLAGS + ["pháp lý"]
# Thứ tự ưu tiên khi các mô hình ngang nhau: đơn giản, ít siêu tham số trước (giống prediction server)
SIMPLICITY_ORDER = ["Extra Trees", "Random Forest", "HistGradientBoosting", "XGBoost"]
print(f"Cấu hình xong. Đặc trưng cơ bản: {len(BASE_FEATURES)} | bổ sung: {len(EXTRA_FEATURES)}")

## 2. Nạp dữ liệu

Dữ liệu là bảng huấn luyện xuất từ database (`listings`) bằng `db_to_training/build_training_csv.py`.
Loại nhà đất, quận và pháp lý đã được mã hóa thành số; tọa độ nhân 10⁹ (sẽ chia lại ở bước 4).

In [ ]:
raw = pd.read_csv(DATA_PATH)
print(f"Số dòng: {len(raw):,} | Số cột: {raw.shape[1]}")
raw.head()

In [ ]:
raw.info()

## 3. Kiểm tra chất lượng dữ liệu

Trước khi xử lý cần biết dữ liệu **thiếu** ở đâu và **trùng** bao nhiêu để quyết định cách xử lý phù hợp.

In [ ]:
quality = pd.DataFrame({
    "thiếu": raw.isna().sum(),
    "% thiếu": raw.isna().mean() * 100,
    "số giá trị khác nhau": raw.nunique(),
})
display(quality)
print(f"Dòng trùng hoàn toàn: {raw.duplicated().sum():,}")

fig, ax = plt.subplots(figsize=(7, 3))
quality["% thiếu"].sort_values().plot.barh(ax=ax, color="#4c78a8")
ax.set_xlabel("% giá trị thiếu")
ax.set_title("Tỷ lệ thiếu theo cột")
plt.tight_layout()
plt.show()

**Nhận xét.** `mặt tiền`, `số tầng`, `phòng ngủ` thiếu nhiều vì tin đăng không bắt buộc nhập (đất nền không có phòng ngủ,
số tầng...). Đây là thiếu **có ý nghĩa**, nên điền bằng median **tính trên train** thay vì xóa dòng.
`tọa độ x/y` thiếu ít nhưng là thông tin vị trí cốt lõi, và không được phép đoán, nên các tin này bị loại.

## 4. Tiền xử lý cơ bản

0. Bỏ các dòng trùng hoàn toàn ở mọi cột. CSV không có `listing_id` nên không phân biệt được đây là cùng một tin
   bị nạp nhiều lần hay là các căn khác nhau có thông số giống hệt (ví dụ cùng dự án). Chọn bỏ để tránh một bản
   nằm ở train, bản kia ở test; ảnh hưởng nhỏ (dưới 1% số dòng).
1. Chia tọa độ cho 10⁹ (`tọa độ x` là **vĩ độ**, `tọa độ y` là **kinh độ**).
2. Loại tin thiếu tọa độ x hoặc y.
3. Loại tin có tọa độ ngoài khung TP.HCM (tọa độ sai, ví dụ điểm mặc định của website).
4. Giải mã nhãn để biểu đồ dễ đọc; **mô hình vẫn dùng mã số**.

In [ ]:
df = raw.copy()
for col in ("tọa độ x", "tọa độ y"):
    df[col] = df[col] / COORD_SCALE

steps = {"ban đầu": len(df)}
df = df.drop_duplicates()
steps["sau khi bỏ dòng trùng hoàn toàn"] = len(df)
df = df.dropna(subset=["tọa độ x", "tọa độ y"])
steps["sau khi bỏ thiếu tọa độ"] = len(df)
inside = df["tọa độ x"].between(*HCM_LAT) & df["tọa độ y"].between(*HCM_LON)
df = df[inside].reset_index(drop=True)
steps["sau khi bỏ tọa độ ngoài TP.HCM"] = len(df)

pd.Series(steps, name="số dòng").to_frame()

In [ ]:
# Cột nhãn chỉ dùng cho EDA; mô hình dùng mã số
df["tên loại"] = df["loại nhà đất"].map(PROPERTY_TYPES)
df["tên quận"] = df["địa chỉ"].map(DISTRICTS)
assert df["tên loại"].notna().all() and df["tên quận"].notna().all(), "Có mã không nằm trong từ điển nhãn"
df.describe().T

## 5. Phân tích khám phá (EDA)
### 5.1 Biến mục tiêu: giá/m²

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 3.8))
sns.histplot(df[TARGET], bins=60, ax=axs[0], color="#4c78a8")
axs[0].set(title="Giá/m² (gốc) — lệch phải mạnh", xlabel="triệu đồng/m²")
sns.histplot(np.log(df[TARGET]), bins=60, ax=axs[1], color="#f58518")
axs[1].set(title="log(giá/m²) — gần đối xứng", xlabel="log(triệu đồng/m²)")
sns.boxplot(x=df[TARGET], ax=axs[2], color="#54a24b")
axs[2].set(title="Boxplot giá/m² (nhiều điểm xa)", xlabel="triệu đồng/m²")
plt.tight_layout()
plt.show()

print(f"Độ lệch (skew) gốc: {df[TARGET].skew():.2f} | sau log: {np.log(df[TARGET]).skew():.2f}")

Phân phối giá gốc lệch phải mạnh, nên **huấn luyện trên `log(giá/m²)`**: giảm ảnh hưởng của vài tin giá cực cao,
sai số mô hình mang nghĩa *tỷ lệ phần trăm* (phù hợp với giá nhà), và dự đoán luôn dương sau khi `exp`.

### 5.2 Biến số: phân phối

In [ ]:
numeric = ["diện tích", "mặt tiền", "phòng ngủ", "số tầng"]
fig, axs = plt.subplots(2, 4, figsize=(16, 6))
for i, col in enumerate(numeric):
    sns.histplot(df[col].dropna(), bins=50, ax=axs[0, i], color="#4c78a8")
    axs[0, i].set_title(col)
    sns.histplot(np.log1p(df[col].dropna()), bins=50, ax=axs[1, i], color="#f58518")
    axs[1, i].set_title(f"log1p({col})")
plt.tight_layout()
plt.show()

### 5.3 Giá theo loại nhà đất và theo quận

In [ ]:
order = df.groupby("tên loại")[TARGET].median().sort_values().index
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.boxplot(data=df, y="tên loại", x=TARGET, order=order, ax=ax, color="#4c78a8", fliersize=2)
ax.set(xscale="log", xlabel="giá/m² (thang log)", ylabel="", title="Giá/m² theo loại nhà đất")
plt.tight_layout()
plt.show()

by_type = df.groupby("tên loại")[TARGET].agg(["count", "median", "mean", "std"]).sort_values("median")
display(by_type)

In [ ]:
order = df.groupby("tên quận")[TARGET].median().sort_values().index
fig, ax = plt.subplots(figsize=(10, 7))
sns.boxplot(data=df, y="tên quận", x=TARGET, order=order, ax=ax, color="#54a24b", fliersize=2)
ax.set(xscale="log", xlabel="giá/m² (thang log)", ylabel="", title="Giá/m² theo quận/huyện")
plt.tight_layout()
plt.show()

Giá chênh lệch rất lớn giữa các quận (trung tâm cao gấp nhiều lần vùng ven) và giữa các loại hình. Hai biến này
là đặc trưng định danh (category), nhưng cây quyết định xử lý được mã số trực tiếp, nên giữ nguyên mã.

### 5.4 Vị trí địa lý

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 6))
sc = axs[0].scatter(df["tọa độ y"], df["tọa độ x"], c=np.log(df[TARGET]), s=6, cmap="turbo", alpha=0.7)
axs[0].scatter([CENTER_LON], [CENTER_LAT], marker="*", s=250, c="black", label="Trung tâm")
plt.colorbar(sc, ax=axs[0], label="log(giá/m²)")
axs[0].set(xlabel="kinh độ", ylabel="vĩ độ", title="Bản đồ tin rao (màu = log giá/m²)")
axs[0].legend()

hb = axs[1].hexbin(df["tọa độ y"], df["tọa độ x"], C=df[TARGET], reduce_C_function=np.median,
                   gridsize=40, cmap="turbo", mincnt=3)
plt.colorbar(hb, ax=axs[1], label="median giá/m²")
axs[1].set(xlabel="kinh độ", ylabel="vĩ độ", title="Median giá/m² theo ô lục giác")
plt.tight_layout()
plt.show()

## 6. Feature engineering

**Khoảng cách tới trung tâm** (km) theo công thức Haversine: giá nhà thường giảm dần khi ra xa trung tâm,
và một biến liên tục như vậy dễ học hơn cặp (vĩ độ, kinh độ) thô.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Khoảng cách đường tròn lớn (km) giữa hai điểm, nhận mảng hoặc số."""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


df["cách trung tâm"] = haversine_km(df["tọa độ x"], df["tọa độ y"], CENTER_LAT, CENTER_LON)
print(df["cách trung tâm"].describe().to_string())

fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
sns.histplot(df["cách trung tâm"], bins=50, ax=axs[0], color="#4c78a8")
axs[0].set(title="Khoảng cách tới trung tâm", xlabel="km")
axs[1].scatter(df["cách trung tâm"], df[TARGET], s=5, alpha=0.25, color="#4c78a8")
axs[1].set(yscale="log", xlabel="km", ylabel="giá/m² (log)", title="Giá/m² giảm khi ra xa trung tâm")
plt.tight_layout()
plt.show()

### 6.1 Tương quan

In [ ]:
corr_cols = [TARGET, "diện tích", "mặt tiền", "phòng ngủ", "số tầng", "cách trung tâm", "tọa độ x", "tọa độ y"]
corr = df[corr_cols].assign(**{TARGET: np.log(df[TARGET])}).corr(method="spearman")
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title("Tương quan Spearman (giá/m² ở dạng log)")
plt.tight_layout()
plt.show()

Dùng **Spearman** vì dữ liệu lệch và quan hệ không tuyến tính hoàn toàn. `cách trung tâm` tương quan âm rõ với giá,
còn các biến quy mô (diện tích, phòng ngủ, số tầng) tương quan yếu hơn: giá/m² phụ thuộc vị trí và loại hình nhiều hơn quy mô.

### 6.2 Đặc trưng bổ sung: cờ thiếu dữ liệu và pháp lý

`mặt tiền` thiếu gần một nửa, `số tầng` và `phòng ngủ` thiếu khoảng 30–40%. Điền median thì mô hình không còn biết
ô đó vốn bị thiếu, trong khi việc người đăng **không ghi** mặt tiền có thể đã mang thông tin (ví dụ nhà trong hẻm).
Thêm cờ `thiếu <cột>` (0/1) cho từng cột, tính trên chính dòng đó nên không rò rỉ dữ liệu.

`pháp lý`: 1 = sổ đỏ/sổ hồng, 2 = hợp đồng mua bán; ô trống (chờ sổ, khác, không ghi) mã hóa thành **0 = không rõ**
thay vì điền median, vì "không rõ pháp lý" là một nhóm riêng.

Có dùng các đặc trưng này hay không sẽ được **quyết định bằng cross-validation** ở bước 9.2.

In [ ]:
for col, flag in zip(FILL_COLUMNS, MISSING_FLAGS):
    df[flag] = df[col].isna().astype(int)
df["pháp lý"] = df["pháp lý"].fillna(0).astype(int)

summary = df.groupby("pháp lý")[TARGET].agg(["count", "median"])
summary.index = summary.index.map({0: "0 - không rõ", 1: "1 - sổ đỏ/sổ hồng", 2: "2 - HĐMB"})
display(summary)
display(pd.DataFrame({
    flag: df.groupby(flag)[TARGET].median() for flag in MISSING_FLAGS
}).rename(index={0: "có dữ liệu", 1: "thiếu"}).T.rename_axis("median giá/m²"))

## 7. Chia train / test

Chia **phân tầng theo khoảng cách trung tâm** (4 nhóm: 0–5, 5–10, 10–15, >15 km) để cả hai tập có cùng cơ cấu
vị trí. Tỷ lệ 80/20, `random_state` cố định để tái lập kết quả.

In [ ]:
df["nhóm khoảng cách"] = pd.cut(df["cách trung tâm"], bins=[0, 5, 10, 15, np.inf],
                                 labels=["0-5km", "5-10km", "10-15km", ">15km"], include_lowest=True)
assert df["nhóm khoảng cách"].notna().all()

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(splitter.split(df, df["nhóm khoảng cách"]))
train_full, test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

share = pd.DataFrame({
    "toàn bộ": df["nhóm khoảng cách"].value_counts(normalize=True),
    "train": train_full["nhóm khoảng cách"].value_counts(normalize=True),
    "test": test["nhóm khoảng cách"].value_counts(normalize=True),
}).sort_index()
display((share * 100).round(1).rename(columns=lambda c: f"{c} (%)"))
print(f"Train: {len(train_full):,} | Test: {len(test):,}")

## 8. Xử lý ngoại lai (chỉ trên train)

So sánh hai cách:

- **IQR (1,5×)**: tự động theo thống kê, nhưng với dữ liệu bất động sản lệch phải sẽ cắt luôn nhiều tin hợp lệ
  (biệt thự, nhà lớn).
- **Luật nghiệp vụ**: chỉ loại giá trị bất khả thi hoặc nằm ngoài phạm vi bài toán.

Ngưỡng/median chỉ tính trên **train**; test giữ nguyên.

In [ ]:
def iqr_mask(frame, columns, k=1.5):
    """True với dòng nằm trong [Q1 - k*IQR, Q3 + k*IQR] ở mọi cột (bỏ qua NaN)."""
    keep = pd.Series(True, index=frame.index)
    for col in columns:
        q1, q3 = frame[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        keep &= frame[col].isna() | frame[col].between(q1 - k * iqr, q3 + k * iqr)
    return keep


# Luật nghiệp vụ (ngưỡng đồng bộ với script huấn luyện của backend)
MAX_BEDROOMS_EXCLUSIVE, MAX_FRONTAGE, MAX_PRICE_PER_M2, AREA_RANGE = 11, 30, 500, (15, 500)

def business_mask(frame):
    return (
        (frame["phòng ngủ"].isna() | (frame["phòng ngủ"] < MAX_BEDROOMS_EXCLUSIVE))
        & (frame["mặt tiền"].isna() | (frame["mặt tiền"] <= MAX_FRONTAGE))
        & (frame[TARGET] <= MAX_PRICE_PER_M2)
        & frame["diện tích"].between(*AREA_RANGE)
    )


iqr_cols = ["diện tích", TARGET, "số tầng", "phòng ngủ", "mặt tiền"]
comparison = pd.DataFrame({
    "giữ lại (IQR)": [iqr_mask(train_full, iqr_cols).sum()],
    "giữ lại (luật nghiệp vụ)": [business_mask(train_full).sum()],
    "tổng train": [len(train_full)],
}, index=["số dòng"]).T
comparison["%"] = comparison["số dòng"] / len(train_full) * 100
display(comparison)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (title, frame) in zip(axs, [
    ("Train gốc", train_full),
    ("Sau IQR", train_full[iqr_mask(train_full, iqr_cols)]),
    ("Sau luật nghiệp vụ", train_full[business_mask(train_full)]),
]):
    ax.scatter(frame["diện tích"], frame[TARGET], s=5, alpha=0.3, color="#4c78a8")
    ax.set(title=f"{title} (n={len(frame):,})", xlabel="diện tích (m²)", yscale="log")
axs[0].set_ylabel("giá/m² (log)")
plt.tight_layout()
plt.show()

**Quyết định: dùng luật nghiệp vụ.** IQR loại nhiều tin hơn rõ rệt (bảng trên, khoảng 21% so với 7% của luật nghiệp vụ), phần lớn là tin hợp lệ (đặc biệt nhà/biệt thự diện tích lớn đang được rao thật
ở TP.HCM), khiến mô hình ít được học về nhóm này. Luật nghiệp vụ giữ nhiều dữ liệu hơn mà vẫn loại các giá trị vô lý.

In [ ]:
# Điền thiếu bằng median của TRAIN, rồi loại ngoại lai chỉ trên train
train_medians = train_full[FILL_COLUMNS].median()
print("Median dùng để điền (từ train):")
display(train_medians.to_frame("median"))

train = train_full.fillna(train_medians)
train = train[business_mask(train)].reset_index(drop=True)
print(f"Train sau xử lý: {len(train):,} / {len(train_full):,} dòng ({len(train) / len(train_full):.1%})")
print(f"Test giữ nguyên: {len(test):,} dòng")

# So sánh mô hình bằng bộ đặc trưng cơ bản; bước 9.2 quyết định có thêm EXTRA_FEATURES hay không
FEATURES = BASE_FEATURES
X_train, y_train = train[FEATURES], np.log(train[TARGET])
X_test, y_test_log, y_test = test[FEATURES], np.log(test[TARGET]), test[TARGET]

## 9. Huấn luyện và so sánh mô hình

Mọi mô hình được bọc trong `Pipeline` với `SimpleImputer(median)` để xử lý ô thiếu của tập test/dữ liệu mới
(median học từ train).

Mô hình tuyến tính và KNN cần thêm hai bước để so sánh công bằng: **one-hot** cho `loại nhà đất` và `địa chỉ` (đây là mã
định danh, không có thứ tự: "quận 23" không lớn hơn "quận 0") và `StandardScaler` cho các cột số. Mô hình cây chia nhánh
được trên mã số nên giữ nguyên.

Đánh giá bằng **5-fold cross-validation trên train** (chọn mô hình, không đụng test). Chỉ số trên thang log:
RMSE/MAE log xấp xỉ sai số tương đối.

In [ ]:
CATEGORICAL = ["loại nhà đất", "địa chỉ"]


def numeric_columns(frame):
    """Các cột còn lại (không phải mã định danh); hàm có tên để pipeline lưu được bằng joblib."""
    return [col for col in frame.columns if col not in CATEGORICAL]


def make_pipeline(model, scale=False):
    if not scale:  # mô hình cây: điền thiếu rồi dùng thẳng mã số
        return Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", model)])
    encode = ColumnTransformer([
        ("onehot", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
        ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
         numeric_columns),
    ])
    return Pipeline([("encode", encode), ("model", model)])


candidates = {
    "Linear Regression": make_pipeline(LinearRegression(), scale=True),
    "KNN": make_pipeline(KNeighborsRegressor(n_neighbors=10, weights="distance"), scale=True),
    "Decision Tree": make_pipeline(DecisionTreeRegressor(max_depth=14, min_samples_leaf=3, random_state=SEED)),
    "Random Forest": make_pipeline(RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=SEED)),
    "Extra Trees": make_pipeline(ExtraTreesRegressor(n_estimators=300, n_jobs=-1, random_state=SEED)),
    "HistGradientBoosting": make_pipeline(HistGradientBoostingRegressor(max_iter=400, learning_rate=0.06, random_state=SEED)),
    "XGBoost": make_pipeline(XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6, subsample=0.8,
                                           colsample_bytree=0.9, n_jobs=-1, random_state=SEED)),
}

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {"r2": "r2", "rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error"}
rows = []
for name, pipe in candidates.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    rows.append({
        "mô hình": name,
        "R² (log)": res["test_r2"].mean(), "± R²": res["test_r2"].std(),
        "RMSE (log)": -res["test_rmse"].mean(), "MAE (log)": -res["test_mae"].mean(),
        "thời gian fit (s)": res["fit_time"].mean(),
    })
cv_table = pd.DataFrame(rows).set_index("mô hình").sort_values("R² (log)", ascending=False)
display(cv_table)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
cv_table["R² (log)"].sort_values().plot.barh(ax=axs[0], xerr=cv_table["± R²"].reindex(cv_table["R² (log)"].sort_values().index),
                                              color="#4c78a8")
axs[0].set(title="R² trên log(giá/m²) — CV 5 fold", xlabel="R²")
cv_table["RMSE (log)"].sort_values(ascending=False).plot.barh(ax=axs[1], color="#f58518")
axs[1].set(title="RMSE trên log(giá/m²) — CV 5 fold (thấp hơn là tốt)", xlabel="RMSE")
plt.tight_layout()
plt.show()

### 9.1 Chọn mô hình: quy tắc "một sai số chuẩn"

Các mô hình cây đứng đầu thường chênh nhau ít hơn độ dao động giữa các fold, tức là **không phân biệt được** về mặt
thống kê. Chọn mô hình có điểm cao nhất trong trường hợp đó là chọn theo nhiễu.

Quy tắc dùng ở đây (one-standard-error rule): ngưỡng = R² cao nhất − **sai số chuẩn** của nó, với sai số chuẩn =
độ lệch chuẩn giữa các fold / √k (k = 5 fold). Trong các mô hình đạt ngưỡng, chọn mô hình
**đơn giản nhất** theo thứ tự `Extra Trees → Random Forest → HistGradientBoosting → XGBoost` (ít siêu tham số, khó overfit,
cùng thứ tự ưu tiên với prediction server của dự án).

In [ ]:
top = cv_table.iloc[0]
standard_error = top["± R²"] / np.sqrt(cv.n_splits)
threshold = top["R² (log)"] - standard_error
tied = cv_table[cv_table["R² (log)"] >= threshold]
print(f"Cao nhất: {cv_table.index[0]} (R² = {top['R² (log)']:.4f}, sai số chuẩn {standard_error:.4f}) -> ngưỡng {threshold:.4f}")
print("Ngang nhau trong 1 sai số chuẩn:", ", ".join(tied.index))

best_name = next((name for name in SIMPLICITY_ORDER if name in tied.index), cv_table.index[0])
print("Mô hình được chọn:", best_name)

### 9.2 Có nên thêm cờ thiếu dữ liệu và pháp lý?

So sánh **cùng mô hình, cùng 5 fold** giữa bộ đặc trưng cơ bản và bộ mở rộng (cơ bản + cờ thiếu + pháp lý). Vì cùng fold
nên so được từng cặp: chỉ dùng bộ mở rộng khi nó giảm RMSE ở **ít nhất 4/5 fold** (cải thiện ổn định, không phải may mắn).

In [ ]:
EXTENDED = BASE_FEATURES + EXTRA_FEATURES
fold_rmse = {}
for label, cols in {"cơ bản": BASE_FEATURES, "mở rộng": EXTENDED}.items():
    res = cross_validate(candidates[best_name], train[cols], y_train, cv=cv,
                         scoring="neg_root_mean_squared_error", n_jobs=1)
    fold_rmse[label] = -res["test_score"]

ablation = pd.DataFrame(fold_rmse, index=[f"fold {i + 1}" for i in range(cv.n_splits)])
ablation["chênh (mở rộng − cơ bản)"] = ablation["mở rộng"] - ablation["cơ bản"]
ablation.loc["trung bình"] = ablation.mean()
display(ablation)

wins = int((ablation["chênh (mở rộng − cơ bản)"].iloc[:-1] < 0).sum())
use_extended = wins >= 4
FEATURES = EXTENDED if use_extended else BASE_FEATURES
X_train, X_test = train[FEATURES], test[FEATURES]
print(f"Bộ mở rộng tốt hơn ở {wins}/{cv.n_splits} fold -> dùng bộ {'mở rộng' if use_extended else 'cơ bản'} "
      f"({len(FEATURES)} đặc trưng)")

**Đọc kết quả.** Bảng ở bước 6.2 cho thấy median giá có khác nhau giữa nhóm thiếu và có dữ liệu, nhưng khác biệt đó phần lớn
đã được giải thích bởi loại nhà đất (ví dụ căn hộ thường không ghi số tầng). Khi đặc trưng mới không mang thêm tín hiệu,
thêm cột chỉ tạo thêm chỗ để mô hình chia nhánh theo nhiễu, nên sai số tăng. Một đặc trưng "nghe hợp lý" vẫn cần được
kiểm chứng bằng CV trước khi đưa vào mô hình.

## 10. Tinh chỉnh siêu tham số

Tinh chỉnh bằng `RandomizedSearchCV` (5-fold, chỉ trên train) cho mô hình đã chọn, với bộ đặc trưng đã chọn.
Lưới tìm kiếm theo họ mô hình; mô hình không có lưới thì dùng cấu hình mặc định ở bước 9.

In [ ]:
param_spaces = {
    "Random Forest": {"model__n_estimators": [200, 400, 600], "model__max_depth": [None, 20, 30],
                      "model__min_samples_leaf": [1, 2, 3, 5], "model__max_features": [0.5, 0.7, 1.0]},
    "Extra Trees": {"model__n_estimators": [300, 500, 800], "model__max_depth": [None, 20, 30],
                    "model__min_samples_leaf": [1, 2, 3, 5], "model__max_features": [0.5, 0.7, 1.0]},
    "XGBoost": {"model__n_estimators": [300, 500, 800], "model__learning_rate": [0.03, 0.05, 0.08],
                "model__max_depth": [4, 6, 8], "model__subsample": [0.7, 0.85, 1.0],
                "model__colsample_bytree": [0.7, 0.9, 1.0]},
    "HistGradientBoosting": {"model__max_iter": [300, 500, 800], "model__learning_rate": [0.03, 0.06, 0.1],
                             "model__max_depth": [None, 6, 10], "model__min_samples_leaf": [10, 20, 40]},
}

if best_name in param_spaces:
    search = RandomizedSearchCV(candidates[best_name], param_spaces[best_name], n_iter=12, cv=cv,
                                scoring="neg_root_mean_squared_error", random_state=SEED, n_jobs=1, refit=True)
    search.fit(X_train, y_train)
    final_model = search.best_estimator_
    print(f"Tham số tốt nhất: {search.best_params_}")
    print(f"RMSE (log) CV: {-search.best_score_:.4f} | trước tinh chỉnh: {ablation.loc['trung bình', 'mở rộng' if use_extended else 'cơ bản']:.4f}")
else:
    final_model = candidates[best_name].fit(X_train, y_train)
    print(f"{best_name}: không có lưới tinh chỉnh, dùng cấu hình mặc định.")

### 10.1 Chẩn đoán: overfit hay thiếu dữ liệu?

- **Sai số train so với CV**: train rất thấp mà CV cao là dấu hiệu mô hình "học thuộc" dữ liệu train (overfit).
- **Learning curve**: huấn luyện với 10% → 100% train. Nếu đường CV vẫn còn đi xuống ở cuối, thêm dữ liệu sẽ còn giúp;
  nếu đã phẳng, muốn tốt hơn phải có thêm đặc trưng chứ không phải thêm dòng.

In [ ]:
train_rmse = float(np.sqrt(mean_squared_error(y_train, final_model.predict(X_train))))
sizes, train_scores, cv_scores = learning_curve(
    candidates[best_name], X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error",
    train_sizes=[0.1, 0.3, 0.55, 0.8, 1.0], n_jobs=1)
train_curve, cv_curve = -train_scores.mean(axis=1), -cv_scores.mean(axis=1)
print(f"RMSE (log) trên train: {train_rmse:.4f} | CV: {cv_curve[-1]:.4f} | chênh {cv_curve[-1] - train_rmse:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sizes, train_curve, "o-", label="train", color="#4c78a8")
ax.plot(sizes, cv_curve, "o-", label="cross-validation", color="#f58518")
ax.fill_between(sizes, cv_curve - cv_scores.std(axis=1), cv_curve + cv_scores.std(axis=1), alpha=0.2, color="#f58518")
ax.set(xlabel="số dòng train", ylabel="RMSE (log)", title=f"Learning curve — {best_name}")
ax.legend()
plt.tight_layout()
plt.show()
print(f"CV RMSE giảm {cv_curve[-2] - cv_curve[-1]:.4f} khi đi từ 80% lên 100% dữ liệu train")

## 11. Đánh giá cuối trên tập test

Tập test chỉ được dùng **một lần** ở bước này. Báo cáo cả thang log lẫn thang thật (sau `exp`, đơn vị triệu đồng/m²),
trên hai phạm vi:

- **Test đầy đủ**: giữ nguyên mọi tin, kể cả tin giá/diện tích cực đoan. Đây là con số chính thức.
- **Test trong phạm vi huấn luyện**: chỉ giữ tin thỏa cùng luật nghiệp vụ ở bước 8 (điền thiếu bằng median của train
  trước khi xét). Cho biết mô hình làm tốt đến đâu với loại tin mà nó được học. Lưu ý luật có xét `giá/m2` nên đây là
  số tham khảo, không thay được con số chính thức.

**Đọc chỉ số nào?** R² và RMSE trên giá thật bị chi phối bởi vài tin giá rất cao (một tin lệch 1.000 triệu/m² đóng góp
vào RMSE bằng hàng nghìn tin lệch 10). Để đánh giá chất lượng chung, ưu tiên **R² (log)**, **MAE** và **Median APE**
(một nửa số tin có sai số tương đối nhỏ hơn con số này).

In [ ]:
def score(pred, y_true):
    """Các chỉ số trên giá thật (pred, y_true: triệu/m²) và trên log."""
    ape = np.abs(pred - y_true) / y_true * 100
    return {
        "R² (log)": r2_score(np.log(y_true), np.log(pred)),
        "R² (giá thật)": r2_score(y_true, pred),
        "RMSE (triệu/m²)": float(np.sqrt(mean_squared_error(y_true, pred))),
        "MAE (triệu/m²)": mean_absolute_error(y_true, pred),
        "MAPE (%)": float(np.mean(ape)),
        "Median APE (%)": float(np.median(ape)),
        "số tin": len(y_true),
    }

# Baseline: luôn dự đoán median giá/m² theo (quận, loại) tính trên train
group_median = train.groupby(["địa chỉ", "loại nhà đất"])[TARGET].median()
district_median = train.groupby("địa chỉ")[TARGET].median()
baseline_pred = np.array([
    group_median.get((d, t), district_median.get(d, train[TARGET].median()))
    for d, t in zip(test["địa chỉ"], test["loại nhà đất"])
])
model_pred = np.exp(final_model.predict(X_test))
in_scope = business_mask(test.fillna(train_medians)).to_numpy()

final_metrics = score(model_pred, y_test)
result = pd.DataFrame({
    ("Test đầy đủ", "Baseline"): score(baseline_pred, y_test),
    ("Test đầy đủ", best_name): final_metrics,
    ("Trong phạm vi huấn luyện", "Baseline"): score(baseline_pred[in_scope], y_test[in_scope]),
    ("Trong phạm vi huấn luyện", best_name): score(model_pred[in_scope], y_test[in_scope]),
})
print(f"Tin test ngoài phạm vi huấn luyện: {(~in_scope).sum():,} / {len(test):,}")
display(result)

### 11.1 Chuyển ngược từ log: giá trung vị hay giá trung bình?

Mô hình học trên `log(giá/m²)`, nên `exp(dự đoán)` ước lượng **giá trung vị** của các căn có cùng đặc điểm, không phải giá
**trung bình** (trung bình luôn cao hơn khi dữ liệu lệch phải). Hiệu chỉnh chuẩn là **hệ số smearing của Duan**:
nhân dự đoán với trung bình của `exp(phần dư)`. Phần dư phải lấy **out-of-fold** (dự đoán trên fold mô hình chưa thấy),
vì phần dư trên chính train của mô hình cây gần như bằng 0.

Notebook giữ **giá trung vị** làm kết quả chính (khớp với chỉ số Median APE và ít bị kéo bởi tin cực đoan) và lưu hệ số
smearing trong file mô hình cho trường hợp cần giá trung bình (ví dụ cộng tổng giá trị nhiều căn).

In [ ]:
oof = cross_val_predict(candidates[best_name], X_train, y_train, cv=cv, n_jobs=1)
smearing = float(np.mean(np.exp(y_train - oof)))
print(f"Hệ số smearing: {smearing:.4f} (giá trung bình ≈ giá trung vị × {smearing:.3f})")
display(pd.DataFrame({
    "giá trung vị (mặc định)": score(model_pred, y_test),
    "giá trung bình (× smearing)": score(model_pred * smearing, y_test),
}))

In [ ]:
pred = model_pred
residual_pct = (pred - y_test) / y_test * 100

fig, axs = plt.subplots(1, 3, figsize=(17, 4.8))
axs[0].scatter(y_test, pred, s=6, alpha=0.3, color="#4c78a8")
lim = [y_test.min(), y_test.max()]
axs[0].plot(lim, lim, "r--", lw=1)
axs[0].set(xscale="log", yscale="log", xlabel="giá/m² thực", ylabel="giá/m² dự đoán", title="Dự đoán vs thực tế (thang log)")

sns.histplot(residual_pct.clip(-100, 200), bins=60, ax=axs[1], color="#f58518")
axs[1].axvline(0, color="black", lw=1)
axs[1].set(xlabel="sai số tương đối (%)", title="Phân phối sai số tương đối (cắt ở -100..200%)")

axs[2].scatter(pred, residual_pct.clip(-100, 200), s=6, alpha=0.3, color="#54a24b")
axs[2].axhline(0, color="black", lw=1)
axs[2].set(xscale="log", xlabel="giá/m² dự đoán", ylabel="sai số tương đối (%)", title="Sai số theo mức giá dự đoán")
plt.tight_layout()
plt.show()

In [ ]:
# Sai số theo từng nhóm: để biết mô hình yếu ở đâu
report = test.assign(pred=pred, ape=np.abs(pred - y_test) / y_test * 100)
by_type = report.groupby("tên loại").agg(số_tin=("ape", "size"), median_APE=("ape", "median"))
by_district = report.groupby("tên quận").agg(số_tin=("ape", "size"), median_APE=("ape", "median"))
by_distance = report.groupby("nhóm khoảng cách", observed=True).agg(số_tin=("ape", "size"), median_APE=("ape", "median"))

fig, axs = plt.subplots(1, 3, figsize=(17, 6), gridspec_kw={"width_ratios": [1.2, 1.2, 0.8]})
for ax, (title, table) in zip(axs, [("loại nhà đất", by_type), ("quận/huyện", by_district), ("khoảng cách", by_distance)]):
    table["median_APE"].sort_values().plot.barh(ax=ax, color="#4c78a8")
    ax.set(title=f"Median APE theo {title} (%)", xlabel="%", ylabel="")
plt.tight_layout()
plt.show()
display(by_type.sort_values("median_APE", ascending=False))

### 11.2 Độ quan trọng đặc trưng

In [ ]:
perm = permutation_importance(final_model, X_test, y_test_log, n_repeats=10, random_state=SEED, n_jobs=1, scoring="r2")
importance = pd.DataFrame({"mean": perm.importances_mean, "std": perm.importances_std}, index=FEATURES).sort_values("mean")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(importance.index, importance["mean"], xerr=importance["std"], color="#4c78a8")
ax.set(title="Permutation importance trên tập test (giảm R² khi xáo trộn cột)", xlabel="giảm R² (log)")
plt.tight_layout()
plt.show()

Permutation importance đo trực tiếp mức giảm chất lượng khi phá vỡ một đặc trưng, tin cậy hơn `feature_importances_`
của cây (vốn thiên về biến có nhiều giá trị). Khi đọc biểu đồ cần lưu ý: các đặc trưng vị trí (`địa chỉ`, tọa độ,
`cách trung tâm`) mang thông tin trùng nhau, nên xáo trộn một cột thì mô hình vẫn lấy được vị trí từ cột khác; độ quan
trọng của **từng** cột vị trí vì vậy thấp hơn đóng góp thật của **cả nhóm** vị trí.

## 12. Lưu mô hình

Lưu `Pipeline` (gồm imputer + mô hình) cùng metadata để tái lập: thứ tự đặc trưng, median điền thiếu, chỉ số test, thời điểm
huấn luyện, hệ số smearing, mã băm SHA-256 của file dữ liệu và phiên bản thư viện (mô hình joblib chỉ nạp an toàn
với đúng phiên bản scikit-learn đã dùng để lưu).
Mô hình dự đoán `log(giá/m²)`, nên khi dùng phải `np.exp` kết quả.

In [ ]:
bundle = {
    "pipeline": final_model,
    "features": FEATURES,
    "target": "log(giá/m2)",
    "train_medians": train_medians.to_dict(),
    "test_metrics": final_metrics,
    "model_name": best_name,
    "trained_at": datetime.now().isoformat(timespec="seconds"),
    "n_train": len(train),
    "n_test": len(test),
    "prediction_is": "median",               # exp(log) = giá trung vị; nhân smearing để ra giá trung bình
    "smearing_factor": smearing,
    "data_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    "library_versions": {lib: importlib.metadata.version(lib)
                         for lib in ("scikit-learn", "xgboost", "numpy", "pandas")},
}
if SAVE_MODEL:
    MODEL_DIR.mkdir(exist_ok=True)
    path = MODEL_DIR / "price_final.joblib"
    joblib.dump(bundle, path)
    print("Đã lưu:", path.resolve())

# Kiểm tra nạp lại và dự đoán một tin mẫu
if SAVE_MODEL:
    loaded = joblib.load(path)
    sample = pd.DataFrame([{
        "loại nhà đất": 2, "địa chỉ": 22, "diện tích": 60, "mặt tiền": np.nan, "phòng ngủ": 3,
        "tọa độ x": 10.80, "tọa độ y": 106.65, "số tầng": 3, "pháp lý": 1,
        "cách trung tâm": float(haversine_km(10.80, 106.65, CENTER_LAT, CENTER_LON)),
    }])
    for col, flag in zip(FILL_COLUMNS, MISSING_FLAGS):   # cờ thiếu tính trước khi imputer điền
        sample[flag] = sample[col].isna().astype(int)
    sample = sample[loaded["features"]]
    print(f"Nhà riêng, Tân Bình, 60 m²: ~{np.exp(loaded['pipeline'].predict(sample))[0]:,.1f} triệu/m²")

## 13. Kết luận và hướng cải thiện

**Kết quả chính** nằm ở bảng ở bước 11: so sánh mô hình cuối với baseline median (quận × loại), cùng sai số theo nhóm.

**Hạn chế.**
- Dữ liệu là **giá rao**, không phải giá giao dịch; thường cao hơn giá thực bán.
- Dữ liệu chỉ từ một nguồn (batdongsan.com.vn) và một khoảng thời gian: mô hình không biết xu hướng thị trường.
- Một số nhóm hiếm (đất nền, kho xưởng, condotel) có rất ít mẫu nên sai số lớn hơn: xem bảng theo loại ở bước 11.
- Không có đặc trưng về chất lượng công trình, hướng nhà, hẻm/mặt tiền thực tế.
- **Chia train/test ngẫu nhiên** nên các tin rất giống nhau (cùng tòa nhà, cùng dự án) có thể nằm ở cả hai phía; số đo
  vì vậy có thể lạc quan hơn so với khi dự đoán ở một khu vực mô hình chưa thấy. Cách đo chặt hơn là `GroupKFold`
  theo phường hoặc theo ô lưới tọa độ.
- Các mô hình cây đứng đầu ngang nhau trong sai số CV; mô hình được chọn theo quy tắc một sai số chuẩn (bước 9.1),
  không có nghĩa nó tốt hơn hẳn các mô hình còn lại.

**Hướng cải thiện.**
1. Thêm đặc trưng tiện ích xung quanh (trường học, bệnh viện, chợ trong bán kính) từ PostGIS.
2. Đặc trưng thời gian (tháng đăng) để bắt xu hướng giá.
3. Mô hình riêng cho từng nhóm loại hình lớn (căn hộ / nhà phố / đất).
4. Dự đoán khoảng (quantile regression hoặc phân vị từ các cây) thay vì một điểm.